In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test.csv
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/train.csv
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test/156155545-Rental-Agreement-Kns-Home.pdf.docx
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test/228094620-Rental-Agreement.pdf.docx
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test/24158401-Rental-Agreement.png
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test/95980236-Rental-Agreement.png
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/train/18325926-Rental-Agreement-1.docx
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/train/36199312-Rental-Agreement.png
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/train/44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/tr

In [2]:
import os

for folder, subfolders, files in os.walk("/kaggle/input"):
    print(folder, "->", len(files), "files")

/kaggle/input -> 0 files
/kaggle/input/datasets -> 0 files
/kaggle/input/datasets/krishrajthakur -> 0 files
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data -> 0 files
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data -> 2 files
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/test -> 4 files
/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data/train -> 10 files


In [3]:
import pandas as pd

DATA = "/kaggle/input/datasets/krishrajthakur/metadata-extraction-data/data"  

train_df = pd.read_csv(f"{DATA}/train.csv", dtype=str)
test_df = pd.read_csv(f"{DATA}/test.csv", dtype=str)
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()
print(train_df.shape)
print(test_df.shape)
print(os.listdir(f"{DATA}/train"))
print(os.listdir(f"{DATA}/test"))

(10, 7)
(4, 7)
['18325926-Rental-Agreement-1.docx', '36199312-Rental-Agreement.png', '44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx', '46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee.docx', '47854715-RENTAL-AGREEMENT.docx', '50070534-RENTAL-AGREEMENT.docx', '54770958-Rental-Agreement.png', '54945838-Rental-Agreement.png', '6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx', '6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx']
['156155545-Rental-Agreement-Kns-Home.pdf.docx', '228094620-Rental-Agreement.pdf.docx', '24158401-Rental-Agreement.png', '95980236-Rental-Agreement.png']


In [4]:
from pathlib import Path

train_dir = Path(DATA) / "train"
test_dir = Path(DATA) / "test"
disk_stems = {p.stem for p in train_dir.iterdir()}   
csv_names = set(train_df["File Name"].str.strip())

print("In CSV but not on disk:", csv_names - disk_stems)
print("On disk but not in CSV:", disk_stems - csv_names)

In CSV but not on disk: {'24158401-Rental-Agreement'}
On disk but not in CSV: {'46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee'}


In [5]:
!pip install -q python-docx
from docx import Document

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00a 0:00:01


In [6]:
def read_docx(path):
    doc = Document(path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            parts.append(" | ".join(cell.text.strip() for cell in row.cells))
    return "\n".join(parts)

In [7]:
! pip install -q easyocr
import easyocr

reader = easyocr.Reader(["en"], gpu = True)

def read_image(path):
    lines = reader.readtext(str(path),detail=0,paragraph=True)
    return "\n".join(lines)

In [8]:
import json
from pathlib import Path

def read_document(path):
    end = Path(path).suffix.lower()
    if end == ".docx":
        return read_docx(path)
    elif end == ".png":
        return read_image(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

def read_folder(df, folder, out_json):
    files = {p.stem.removesuffix(".pdf"): p for p in Path(folder).iterdir()}
    texts = {}
    for name in df["File Name"].str.strip():
        if name in files:
            texts[name] = read_document(files[name])
        else:
            print("skipping:", name)
    json.dump(texts, open(out_json, "w"))
    print(len(texts), "documents read ->", out_json)
    return texts

In [9]:
train_texts = read_folder(train_df, Path(DATA) / "train", "/kaggle/working/train_texts.json")
test_texts  = read_folder(test_df,  Path(DATA) / "test",  "/kaggle/working/test_texts.json")

skipping: 24158401-Rental-Agreement
9 documents read -> /kaggle/working/train_texts.json
4 documents read -> /kaggle/working/test_texts.json


In [10]:
!pip install -q -U bitsandbytes accelerate
import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 13.2 MB/s eta 0:00:00


In [11]:
!pip install -q google-genai
import time
from google import genai
from kaggle_secrets import UserSecretsClient

client = genai.Client(api_key=UserSecretsClient().get_secret("gemini_api_key_ty"))
import time

GEMINI_MODELS = ["gemini-3.8-flash"]

def call_gemini(prompt):
    for model_name in GEMINI_MODELS:
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config={"temperature": 0},
                )
                return response.text, model_name
            except Exception as e:
                print(f"{model_name} attempt {attempt + 1} failed: {str(e)[:80]}")
                time.sleep(5 * (attempt + 1))   # wait 5s, then 10s, then 15s
    raise RuntimeError("All Gemini models failed")

def extract_gemini(text, shots=()):
    parts = [SYSTEM_PROMPT]
    for shot_text, shot_answer in shots:
        parts.append("Document:\n" + shot_text[:SHOT_CHARS])
        parts.append("Answer:\n" + json.dumps(shot_answer, ensure_ascii=False))
    parts.append("Document:\n" + text[:TARGET_CHARS])
    parts.append("Answer:")

    reply, used = call_gemini("\n\n".join(parts))
    print("used:", used)
    start, end = reply.find("{"), reply.rfind("}")
    time.sleep(4)   
    return json.loads(reply[start:end + 1])

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")

In [13]:
SYSTEM_PROMPT = """You extract metadata from rental agreements.
Return ONLY a JSON object with exactly these keys:
"Aggrement Value", "Aggrement Start Date", "Aggrement End Date",
"Renewal Notice (Days)", "Party One", "Party Two".

Rules:
- Dates must be written DD.MM.YYYY (with dots). Convert dates written in words, e.g. "fifth day of December 2008" -> "05.12.2008".
- Aggrement Value is the monthly rent as digits only: no currency symbol, no commas, no decimals. If the digits and the words disagree, use the digits.
- Aggrement End Date: if it is not written, work it out from the start date and the agreement period (e.g. 11 months or 1 year).
- Renewal Notice (Days): the advance notice either party must give before ending, vacating or not renewing (e.g. "one month prior notice" = 30, "three month advance notice" = 90). Return null only if no notice period is mentioned anywhere in the document.
- Party One is the landlord / lessor / owner. Party Two is the tenant / lessee.
- If a date or amount is unreadable or garbled in the document, return null instead of guessing.
- Use only numbers that appear in the document. Never invent an amount.
- For Party One and Party Two, output ONLY the name itself. Always remove these before the name: Mr., Mrs., Ms., Sri, Smt., Shri, Prof, Dr, M/s. Remove everything after "S/o", "D/o", "W/o" or "C/o", and remove age, address and anything in brackets. Keep the name's own capital letters, dots and spacing exactly as written. Examples: "Sri Ramesh Kumar" -> "Ramesh Kumar"; "Mrs. S.Latha" -> "S.Latha"; "Mr. P C MATHEW, S/O K JOSEPH" -> "P C MATHEW"; "SRI ABC TRADERS Private Ltd." -> "ABC TRADERS Private Ltd". If a party has several people, keep them in one string joined as in the document.
- Aggrement End Date: if it is not written, compute it as the start date plus the agreement period, minus one day. Examples: start 01.03.2009 with 11 months -> 31.01.2010; start 10.06.2009 with 12 months -> 09.06.2010.
- Output the JSON and nothing else."""

In [14]:
SHOT_CHARS = 3500     
TARGET_CHARS = 10000  

def extract(text, shots=()):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for shot_text, shot_answer in shots:
        messages.append({"role": "user", "content": "Document:\n" + shot_text[:SHOT_CHARS]})
        messages.append({"role": "assistant", "content": json.dumps(shot_answer, ensure_ascii=False)})
    messages.append({"role": "user", "content": "Document:\n" + text[:TARGET_CHARS]})

    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    del inputs, out
    torch.cuda.empty_cache()

    start, end = reply.find("{"), reply.rfind("}")
    return json.loads(reply[start:end + 1])

In [15]:
FIELDS = ["Aggrement Value", "Aggrement Start Date", "Aggrement End Date",
          "Renewal Notice (Days)", "Party One", "Party Two"]

def norm(x):
    if x is None or x != x:
        return ""
    return str(x).strip()

train_df = pd.read_csv(f"{DATA}/train.csv", dtype=str)
train_df.columns = train_df.columns.str.strip()

cols = [c for c in train_df.columns if c != "File Name"]
a = train_df["File Name"].str.strip() == "54770958-Rental-Agreement"
b = train_df["File Name"].str.strip() == "54945838-Rental-Agreement"
tmp = train_df.loc[a, cols].values.copy()
train_df.loc[a, cols] = train_df.loc[b, cols].values
train_df.loc[b, cols] = tmp

gold = train_df.set_index(train_df["File Name"].str.strip())

In [16]:
POOL = ["36199312-Rental-Agreement", "50070534-RENTAL-AGREEMENT",
        "47854715-RENTAL-AGREEMENT", "54770958-Rental-Agreement"]

def make_shots(target):
    names = [n for n in POOL if n != target][:3]
    return [(train_texts[n], {f: (norm(gold.loc[n, f]) or None) for f in FIELDS}) for n in names]

In [ ]:
preds = {}
for name, text in train_texts.items():
    preds[name] = extract_gemini(text, make_shots(name))

hits = {f: 0 for f in FIELDS}
for name in preds:
    for f in FIELDS:
        predicted = norm(preds[name][f])
        correct   = norm(gold.loc[name, f])
        if predicted == correct:
            hits[f] += 1
        else:
            print("MISS", name, "|", f, "| predicted:", predicted, "| correct:", correct)

print()
for f in FIELDS:
    print(f"{f:25s} {hits[f] / len(preds):.2f}")

In [18]:
test_df = pd.read_csv(f"{DATA}/test.csv", dtype=str)
test_df.columns = test_df.columns.str.strip()
test_gold = test_df.set_index(test_df["File Name"].str.strip())

test_texts = json.load(open("/kaggle/working/test_texts.json"))
shots = make_shots(None)

test_preds = {}
for name, text in test_texts.items():
    test_preds[name] = extract_gemini(text, shots)

out = pd.DataFrame.from_dict(test_preds, orient="index")[FIELDS]
out.index.name = "File Name"
out.to_csv("/kaggle/working/predictions_gemini_3.8_ty.csv")

for f in FIELDS:
    hits = sum(norm(test_preds[n][f]) == norm(test_gold.loc[n, f]) for n in test_preds)
    print(f"{f:25s} {hits / len(test_preds):.2f}")

used: gemini-3.8-flash
used: gemini-3.8-flash
used: gemini-3.8-flash
used: gemini-3.8-flash
Aggrement Value           0.75
Aggrement Start Date      0.75
Aggrement End Date        0.75
Renewal Notice (Days)     1.00
Party One                 1.00
Party Two                 0.25
